# 合法请求通过，越权动作拒绝

> 状态：verified；本地教学 fixture；执行日期：2026-09-06。

本实验不调用模型。我们假设模型已经被外部文本诱导产生一个工具候选，再验证执行层会不会放行。它测的是权限边界，不能测模型真实抗注入能力。

先读[提示注入机制](../01-concepts/01-threat-model-and-injection.md)。A 组用户只能读 A 组记录；删除需要与具体动作绑定的审批；没有任何网络工具。

执行说明：本次因环境禁止 Kernel socket，使用独立 Python 进程内 IPython 按单元顺序执行并保存真实输出；不是 Jupyter kernel 运行。普通 Jupyter 环境可按原顺序运行。

In [1]:
from pathlib import Path
import sys, json
from pprint import pprint
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "10-Knowledge").is_dir())
sys.path.insert(0, str(ROOT / "10-Knowledge/11-safety-security-governance/05-code"))
from policy_lab import Principal, Proposal, Denied, fixture, injection_cases
rt = fixture()
user = Principal("learner", "A", frozenset({"read", "delete"}))
rows = []
for name, untrusted_text, proposal, expected in injection_cases():
    try:
        output = rt.execute(user, "run-1", proposal)
        allowed, reason = True, "allowed"
        assert "fixture-secret-A" not in str(output)
    except Denied as exc:
        allowed, reason = False, str(exc)
    assert allowed == expected, name
    rows.append({"case": name, "expected_allow": expected, "actual_allow": allowed, "reason": reason})
pprint(rows)
assert "a1" in rt.records and "b1" in rt.records


[{'actual_allow': True,
  'case': 'legitimate_read',
  'expected_allow': True,
  'reason': 'allowed'},
 {'actual_allow': True,
  'case': 'benign_quote',
  'expected_allow': True,
  'reason': 'allowed'},
 {'actual_allow': False,
  'case': 'cross_tenant',
  'expected_allow': False,
  'reason': 'resource_not_accessible'},
 {'actual_allow': False,
  'case': 'exfiltration',
  'expected_allow': False,
  'reason': 'unknown_tool'},
 {'actual_allow': False,
  'case': 'unapproved_delete',
  'expected_allow': False,
  'reason': 'approval_required_or_mismatch'},
 {'actual_allow': False,
  'case': 'path_traversal',
  'expected_allow': False,
  'reason': 'resource_not_accessible'}]


正常引用“忽略此前指令”的教材也能够读取，因为策略不靠关键词封禁。非法动作中，跨租户访问被资源权限拦截，上传密钥因工具未注册被拒绝，无审批删除被审批规则拦截。

分别计算合法通过率和非法拒绝率，避免“全部拒绝”看起来像成功。样本只有 2+4 条，指标仅是 fixture 断言的摘要。

In [2]:
legitimate = [r for r in rows if r["expected_allow"]]
illegal = [r for r in rows if not r["expected_allow"]]
allow_rate = sum(r["actual_allow"] for r in legitimate) / len(legitimate)
block_rate = sum(not r["actual_allow"] for r in illegal) / len(illegal)
print({"legitimate_pass": f"{sum(r['actual_allow'] for r in legitimate)}/{len(legitimate)}",
       "illegal_block": f"{sum(not r['actual_allow'] for r in illegal)}/{len(illegal)}",
       "allow_rate": allow_rate, "block_rate": block_rate})
assert allow_rate == block_rate == 1


{'legitimate_pass': '2/2', 'illegal_block': '4/4', 'allow_rate': 1.0, 'block_rate': 1.0}


有删除权限还不等于本次删除已经获准。下面模拟可信服务端收到了用户对 a2 的确认。令牌首先拿去删 a1 应失败，随后按原请求删除 a2 应成功；不会把一次批准扩大成所有对象的删除权限。

In [3]:
token = rt.approve(user, "run-1", Proposal("delete_record", "a2"))
try:
    rt.execute(user, "run-1", Proposal("delete_record", "a1", token))
except Denied as exc:
    print("换对象被拒绝：", exc)
else:
    raise AssertionError("approval not bound to resource")
result = rt.execute(user, "run-1", Proposal("delete_record", "a2", token))
print("合法审批后删除：", result)
assert "a2" not in rt.records and "a1" in rt.records
try:
    rt.execute(user, "run-1", Proposal("delete_record", "a2", token))
except Denied:
    print("重复使用令牌被拒绝")
else:
    raise AssertionError("token reused")


换对象被拒绝： approval_required_or_mismatch
合法审批后删除： {'deleted': 'a2'}
重复使用令牌被拒绝


最后观察审计。日志记录用户、动作、资源和决策，未记录原文、secret 或审批令牌。这个内存审计只能演示字段选择，正式审计还需持久化、访问控制和保留策略。

In [4]:
pprint(rt.audit)
assert "fixture-secret-A" not in json.dumps(rt.audit)
assert token not in json.dumps(rt.audit)
evidence = {"data": "synthetic proposals without a model", "cases": rows,
            "legitimate_allow_rate": allow_rate, "illegal_block_rate": block_rate,
            "approved_delete": result, "remaining_resource_ids": sorted(rt.records), "audit": rt.audit}
artifact = Path("artifacts/policy.json")
artifact.parent.mkdir(exist_ok=True)
artifact.write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding="utf-8")
print("结构化证据：", artifact)


[{'decision': 'allow',
  'resource_id': 'a1',
  'run_id': 'run-1',
  'tool': 'read_record',
  'user': 'learner'},
 {'decision': 'allow',
  'resource_id': 'a1',
  'run_id': 'run-1',
  'tool': 'read_record',
  'user': 'learner'},
 {'decision': 'deny',
  'reason': 'resource_not_accessible',
  'resource_id': 'b1',
  'run_id': 'run-1',
  'tool': 'read_record',
  'user': 'learner'},
 {'decision': 'deny',
  'reason': 'unknown_tool',
  'resource_id': 'a1',
  'run_id': 'run-1',
  'tool': 'send_http',
  'user': 'learner'},
 {'decision': 'deny',
  'reason': 'approval_required_or_mismatch',
  'resource_id': 'a1',
  'run_id': 'run-1',
  'tool': 'delete_record',
  'user': 'learner'},
 {'decision': 'deny',
  'reason': 'resource_not_accessible',
  'resource_id': '../b1',
  'run_id': 'run-1',
  'tool': 'read_record',
  'user': 'learner'},
 {'decision': 'deny',
  'reason': 'approval_required_or_mismatch',
  'resource_id': 'a1',
  'run_id': 'run-1',
  'tool': 'delete_record',
  'user': 'learner'},
 {'dec

已验证：合法读取与合法批准删除成功；跨租户、越权工具、无批准删除和审批参数变更被拒绝。没有验证语言模型是否会生成恶意动作、正文内容脱敏、OAuth、沙箱或生产审计。

练习：给主体去掉 delete scope，解释为什么即使有一段“已获批准”的文本也不能删除；再查看[测试](../05-code/test_policy.py)中审批过期、跨用户和跨 Run 的负例。